In [1]:
import pandas as pd 
import logging 
import duckdb 

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    filename='dataCleaning.log'
)
logger = logging.getLogger(__name__)

In [ ]:
### load duckdb tables into pandas dfs 
con = None
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.")

    # inserting tables 
    cville = con.execute(f"""
        SELECT * FROM cville;
    """).fetchdf()
    dulles = con.execute(f"""
        SELECT * FROM dulles;
    """).fetchdf()
    lynchburg = con.execute(f"""
        SELECT * FROM lynchburg;
    """).fetchdf()
    norfolk = con.execute(f"""
        SELECT * FROM norfolk;
    """).fetchdf()

    logger.info("All tables loaded into pandas dataframes")

except Exception as e:
    logger.error(f"An error occurred: {e}")

In [3]:
# Classify extreme weather events (EWE) based on previous research done 
def add_ewe(df, name):
    # make list of extreme weather codes to check for in the DailyWeather column 
    extreme_codes = ['PL', 'DU', 'HZ', 'BLSN', 'FC', 'WIND', 'SN']

    # iterate through each row and check for extreme weather conditions
    for index, row in df.iterrows():
        # convert to string to check for codes 
        weather = str(row['DailyWeather'])  
        if any(code in weather for code in extreme_codes):
            df.at[index, 'EWE'] = 1

        # too high or too low pressure 
        elif row['DailyAverageStationPressure'] > 30.40 or row['DailyAverageStationPressure'] < 29.40:
            df.at[index, 'EWE'] = 1

        # higher than 95 is extreme temp 
        elif row['DailyMaximumDryBulbTemperature'] > 95:
            df.at[index, 'EWE'] = 1

        # lower than 10 is extreme temp
        elif row['DailyMinimumDryBulbTemperature'] < 10:
            df.at[index, 'EWE'] = 1

        # daily precipitation greater than 2 inches is extreme
        elif row['DailyPrecipitation'] > 2:
            df.at[index, 'EWE'] = 1
        
        # more than 2 inches of snow daily is extreme
        elif row['DailySnowfall'] > 2: 
            df.at[index, 'EWE'] = 1
        
        # sustained wind speed greater than 58 mph is extreme
        elif row['DailySustainedWindSpeed'] > 58:
            df.at[index, 'EWE'] = 1

        # else no extreme weather event
        else:
            df.at[index, 'EWE'] = 0

    logger.info("Extreme weather events classified.")

    print(f"EWE counts for {name}: {df['EWE'].value_counts()}")
    # return df as new df with EWE column added
    return df 

In [4]:
dfs = {'cville': cville, 'dulles': dulles, 'lynchburg': lynchburg, 'norfolk': norfolk}
for name, df in dfs.items():
    logger.info(f"{name} starting")

    df.drop(columns=['Entry_ID', 'REPORT_TYPE', 'SOURCE'], inplace=True)
    
    # list of non-daily columns to keep
    cols_keep = ['STATION', 'DATE', 'Sunrise', 'Sunset']
    # create list of columns to keep by adding daily columns to cols_keep
    cols_keep2 = cols_keep + [col for col in df.columns if 'daily' in col.lower()] 
    # new df of only keeping columns
    df2 = df[cols_keep2].copy() 

    # list of columns to clean
    cols_fixing = ['DailyAverageDewPointTemperature', 'DailyAverageDryBulbTemperature',
        'DailyAverageRelativeHumidity', 'DailyAverageSeaLevelPressure',
        'DailyAverageStationPressure', 'DailyAverageWetBulbTemperature',
        'DailyAverageWindSpeed', 'DailyCoolingDegreeDays',
        'DailyDepartureFromNormalAverageTemperature', 'DailyHeatingDegreeDays',
        'DailyMaximumDryBulbTemperature', 'DailyMinimumDryBulbTemperature',
        'DailyPeakWindDirection', 'DailySustainedWindDirection', 'DailySustainedWindSpeed', 
        'DailyPeakWindSpeed', 'DailyPrecipitation', 'DailySnowfall', 'DailySnowDepth' ]

    # for each column 
    for col in cols_fixing:
        # fill any NaN values with 0 (to be all numerical)
        df2[col] = df2[col].fillna(0)
        # remove 's' from values
        df2[col] = df2[col].astype(str).str.replace('s', '', regex=False)
        # replace 'T' with 0 because represents trace amounts, which will be interpreted as 0 for this analysis 
        df2[col] = df2[col].astype(str).str.strip().replace('T', 0)

        # try to convert column to float 
        try:
            df2[col] = df2[col].astype(float)
            # print(col)
        # if fails, use pd.to_numeric with coercion
        except Exception as e:
            df2[col] = pd.to_numeric(df2[col], errors='coerce').fillna(0)
            # print(f"Column '{col}' had conversion issues, used pd.to_numeric as fallback.")

    # ensure Date is in proper datetime format 
    df2['DATE'] = pd.to_datetime(df2['DATE'], format='%Y-%m-%d') 
    # extract month, day of week, and day of year for easier analysis later on
    df2['Month'] = df2['DATE'].dt.month
    df2['DayOfWeek'] = df2['DATE'].dt.dayofweek
    df2['DayOfYear'] = df2['DATE'].dt.dayofyear

    # add ewe column and save as new df3
    df3 = add_ewe(df2, name)

    # save to parquet file for later use
    df3.to_parquet(f"data/{name}_cleaned.parquet", engine='pyarrow', index=False)

    # save to duckdb table
    with duckdb.connect(database='project1.db', read_only=False):
        con.execute(f"""
        DROP TABLE IF EXISTS {name};
        CREATE TABLE {name} AS
        SELECT * FROM read_parquet('data/{name}_cleaned.parquet');
        """)

    logger.info(f"{name} has {df3.shape[0]} rows cleaned for {df3.shape[1]} columns")


con.close() 

EWE counts for cville: EWE
1.0    3194
0.0    1537
Name: count, dtype: int64
EWE counts for dulles: EWE
0.0    3757
1.0     992
Name: count, dtype: int64
EWE counts for lynchburg: EWE
1.0    4617
0.0     125
Name: count, dtype: int64
EWE counts for norfolk: EWE
0.0    4128
1.0     621
Name: count, dtype: int64
